In [1]:
//Iniciando o Spark
import $ivy.`org.apache.spark::spark-sql:3.5.0`

import org.apache.spark.sql.SparkSession

// Inicializa o Spark em modo Local utilizando 2 threads para simular os Workers
val spark = SparkSession.builder()
  .appName("AulaSparkScalaDemo")
  .master("local[2]")
  .getOrCreate()

import spark.implicits._
spark.sparkContext.setLogLevel("ERROR")

val sc = spark.sparkContext

println(s"Driver App Name: ${sc.appName}")
println(s"Modo de Execução (Master): ${sc.master}")
println(s"Spark UI disponível em: ${sc.uiWebUrl.getOrElse("http://localhost:4040")}")

Failed to download https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.4/hadoop-client-runtime-3.3.4-sources.jar
Failed to download https://repo1.maven.org/maven2/io/netty/netty-all/4.1.96.Final/netty-all-4.1.96.Final-sources.jar
Failed to download https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.4/hadoop-client-runtime-3.3.4-sources.jar.sha1
Failed to download https://repo1.maven.org/maven2/io/netty/netty-all/4.1.96.Final/netty-all-4.1.96.Final-sources.jar.sha1
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/13 19:58:30 WARN Utils: Your hostname, MacBook-Pro-de-Larissa.local resolves to a loopback address: 127.0.0.1; using 191.52.71.206 instead (on interface en0)
26/11/13 19:58:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/11/13 19:58:30 INFO SparkContext: Running Spark version 3.5.0
26/11/13 19:58:30 INFO SparkContext: OS info Mac OS X, 15.7.3, aarch64
26/11/13 19:58:

Driver App Name: AulaSparkScalaDemo
Modo de Execução (Master): local[2]
Spark UI disponível em: http://191.52.71.206:4040


import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.SparkSession@3c0e2091
import spark.implicits._
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@39d251ca

In [2]:
import org.apache.spark.sql.{DataFrame, Dataset, Row}
import org.apache.spark.sql.functions._

// Exemplo de leitura de CSV
 val dfVendasCSV = spark.read
   .option("header", "true")
   .option("inferSchema", "true")
   .csv("vendas.csv")

dfVendasCSV.printSchema()
dfVendasCSV.show()

root
 |-- id_venda: integer (nullable = true)
 |-- cliente: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- data_venda: date (nullable = true)

+--------+--------------+----------------+-------+------+----------+
|id_venda|       cliente|       categoria|  valor|estado|data_venda|
+--------+--------------+----------------+-------+------+----------+
|       1| Gabriela Lima|          Livros|3634.52|    MG|2024-03-14|
|       2| Isabela Pires|       Alimentos| 1245.1|    SP|2024-11-02|
|       3|     Ana Silva|     Eletrônicos| 4812.9|    PR|2024-07-22|
|       4|  Hugo Martins|          Roupas| 890.15|    RJ|2024-01-10|
|       5|    Carla Dias|Casa e Decoração|  210.4|    SC|2024-09-05|
|       6|   Bruno Souza|         Higiene|   45.9|    BA|2024-05-18|
|       7|   Diego Alves|     Eletrônicos| 2890.0|    SP|2024-02-11|
|       8|   Elena Costa|          Roupas|  120.5|    MG|2024-08-30|

import org.apache.spark.sql.{DataFrame, Dataset, Row}
import org.apache.spark.sql.functions._
dfVendasCSV: DataFrame = [id_venda: int, cliente: string ... 4 more fields]

In [5]:
val resultado = dfVendasCSV
      .filter($"valor" > (24 * 60))          
      .filter($"cliente" === 50)  
      .select("categoria", "estado", "valor")   

resultado.explain(true)

== Parsed Logical Plan ==
'Project ['categoria, 'estado, 'valor]
+- Filter (cast(cliente#18 as int) = 50)
   +- Filter (valor#20 > cast(1440 as double))
      +- Relation [id_venda#17,cliente#18,categoria#19,valor#20,estado#21,data_venda#22] csv

== Analyzed Logical Plan ==
categoria: string, estado: string, valor: double
Project [categoria#19, estado#21, valor#20]
+- Filter (cast(cliente#18 as int) = 50)
   +- Filter (valor#20 > cast(1440 as double))
      +- Relation [id_venda#17,cliente#18,categoria#19,valor#20,estado#21,data_venda#22] csv

== Optimized Logical Plan ==
Project [categoria#19, estado#21, valor#20]
+- Filter ((isnotnull(valor#20) AND isnotnull(cliente#18)) AND ((valor#20 > 1440.0) AND (cast(cliente#18 as int) = 50)))
   +- Relation [id_venda#17,cliente#18,categoria#19,valor#20,estado#21,data_venda#22] csv

== Physical Plan ==
*(1) Project [categoria#19, estado#21, valor#20]
+- *(1) Filter (((isnotnull(valor#20) AND isnotnull(cliente#18)) AND (valor#20 > 1440.0)) AND (c

resultado: DataFrame = [categoria: string, estado: string ... 1 more field]

In [6]:
//Tabela de Vendas com 1M de linhas
val dfVendas = spark.range(1, 1000000)
      .withColumnRenamed("id", "venda_id")
      .withColumn("cliente_id", ($"venda_id" % 500) + 1)
      .withColumn("valor", $"venda_id" * 2.5)

//Tabela de Clientes com 500
val dfClientes = spark.range(1, 501)
      .withColumnRenamed("id", "cliente_id")
      .withColumn("nome", concat(lit("Cliente_"), $"cliente_id"))

dfVendas.show()
dfClientes.show()

+--------+----------+-----+
|venda_id|cliente_id|valor|
+--------+----------+-----+
|       1|         2|  2.5|
|       2|         3|  5.0|
|       3|         4|  7.5|
|       4|         5| 10.0|
|       5|         6| 12.5|
|       6|         7| 15.0|
|       7|         8| 17.5|
|       8|         9| 20.0|
|       9|        10| 22.5|
|      10|        11| 25.0|
|      11|        12| 27.5|
|      12|        13| 30.0|
|      13|        14| 32.5|
|      14|        15| 35.0|
|      15|        16| 37.5|
|      16|        17| 40.0|
|      17|        18| 42.5|
|      18|        19| 45.0|
|      19|        20| 47.5|
|      20|        21| 50.0|
+--------+----------+-----+
only showing top 20 rows

+----------+----------+
|cliente_id|      nome|
+----------+----------+
|         1| Cliente_1|
|         2| Cliente_2|
|         3| Cliente_3|
|         4| Cliente_4|
|         5| Cliente_5|
|         6| Cliente_6|
|         7| Cliente_7|
|         8| Cliente_8|
|         9| Cliente_9|
|        10|Cl

dfVendas: DataFrame = [venda_id: bigint, cliente_id: bigint ... 1 more field]
dfClientes: DataFrame = [cliente_id: bigint, nome: string]

In [7]:
//Broadcast Hash Join - dfClientes é suficientemente pequeno
val resultado = dfVendas.join(dfClientes, "cliente_id")
      .filter($"valor" > (24 * 60))        
      .select("venda_id", "nome") 

resultado.explain(true)

== Parsed Logical Plan ==
'Project ['venda_id, 'nome]
+- Filter (valor#74 > cast(1440 as double))
   +- Project [cliente_id#71L, venda_id#69L, valor#74, nome#82]
      +- Join Inner, (cliente_id#71L = cliente_id#80L)
         :- Project [venda_id#69L, cliente_id#71L, (cast(venda_id#69L as double) * 2.5) AS valor#74]
         :  +- Project [venda_id#69L, ((venda_id#69L % cast(500 as bigint)) + cast(1 as bigint)) AS cliente_id#71L]
         :     +- Project [id#67L AS venda_id#69L]
         :        +- Range (1, 1000000, step=1, splits=Some(2))
         +- Project [cliente_id#80L, concat(Cliente_, cast(cliente_id#80L as string)) AS nome#82]
            +- Project [id#78L AS cliente_id#80L]
               +- Range (1, 501, step=1, splits=Some(2))

== Analyzed Logical Plan ==
venda_id: bigint, nome: string
Project [venda_id#69L, nome#82]
+- Filter (valor#74 > cast(1440 as double))
   +- Project [cliente_id#71L, venda_id#69L, valor#74, nome#82]
      +- Join Inner, (cliente_id#71L = cliente

resultado: DataFrame = [venda_id: bigint, nome: string]

In [8]:
val dfBroadcast = dfVendas.join(dfClientes, "cliente_id")
dfBroadcast.explain(extended=true)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [cliente_id])
:- Project [venda_id#69L, cliente_id#71L, (cast(venda_id#69L as double) * 2.5) AS valor#74]
:  +- Project [venda_id#69L, ((venda_id#69L % cast(500 as bigint)) + cast(1 as bigint)) AS cliente_id#71L]
:     +- Project [id#67L AS venda_id#69L]
:        +- Range (1, 1000000, step=1, splits=Some(2))
+- Project [cliente_id#80L, concat(Cliente_, cast(cliente_id#80L as string)) AS nome#82]
   +- Project [id#78L AS cliente_id#80L]
      +- Range (1, 501, step=1, splits=Some(2))

== Analyzed Logical Plan ==
cliente_id: bigint, venda_id: bigint, valor: double, nome: string
Project [cliente_id#71L, venda_id#69L, valor#74, nome#82]
+- Join Inner, (cliente_id#71L = cliente_id#80L)
   :- Project [venda_id#69L, cliente_id#71L, (cast(venda_id#69L as double) * 2.5) AS valor#74]
   :  +- Project [venda_id#69L, ((venda_id#69L % cast(500 as bigint)) + cast(1 as bigint)) AS cliente_id#71L]
   :     +- Project [id#67L AS venda_id#69L]
   :       

dfBroadcast: DataFrame = [cliente_id: bigint, venda_id: bigint ... 2 more fields]

In [9]:
val dfBroadcast_explicit = dfVendas.join(broadcast(dfClientes), "cliente_id")
dfBroadcast_explicit.explain(extended=true)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [cliente_id])
:- Project [venda_id#69L, cliente_id#71L, (cast(venda_id#69L as double) * 2.5) AS valor#74]
:  +- Project [venda_id#69L, ((venda_id#69L % cast(500 as bigint)) + cast(1 as bigint)) AS cliente_id#71L]
:     +- Project [id#67L AS venda_id#69L]
:        +- Range (1, 1000000, step=1, splits=Some(2))
+- ResolvedHint (strategy=broadcast)
   +- Project [cliente_id#80L, concat(Cliente_, cast(cliente_id#80L as string)) AS nome#82]
      +- Project [id#78L AS cliente_id#80L]
         +- Range (1, 501, step=1, splits=Some(2))

== Analyzed Logical Plan ==
cliente_id: bigint, venda_id: bigint, valor: double, nome: string
Project [cliente_id#71L, venda_id#69L, valor#74, nome#82]
+- Join Inner, (cliente_id#71L = cliente_id#80L)
   :- Project [venda_id#69L, cliente_id#71L, (cast(venda_id#69L as double) * 2.5) AS valor#74]
   :  +- Project [venda_id#69L, ((venda_id#69L % cast(500 as bigint)) + cast(1 as bigint)) AS cliente_id#71L]
   :     +

dfBroadcast_explicit: DataFrame = [cliente_id: bigint, venda_id: bigint ... 2 more fields]

In [10]:
val dfBroadcast_CSV = dfVendasCSV.join(dfClientes, dfVendasCSV.col("cliente") === dfClientes.col("cliente_id"))
dfBroadcast_CSV.explain(extended=true)

== Parsed Logical Plan ==
Join Inner, (cast(cliente#18 as bigint) = cliente_id#80L)
:- Relation [id_venda#17,cliente#18,categoria#19,valor#20,estado#21,data_venda#22] csv
+- Project [cliente_id#80L, concat(Cliente_, cast(cliente_id#80L as string)) AS nome#82]
   +- Project [id#78L AS cliente_id#80L]
      +- Range (1, 501, step=1, splits=Some(2))

== Analyzed Logical Plan ==
id_venda: int, cliente: string, categoria: string, valor: double, estado: string, data_venda: date, cliente_id: bigint, nome: string
Join Inner, (cast(cliente#18 as bigint) = cliente_id#80L)
:- Relation [id_venda#17,cliente#18,categoria#19,valor#20,estado#21,data_venda#22] csv
+- Project [cliente_id#80L, concat(Cliente_, cast(cliente_id#80L as string)) AS nome#82]
   +- Project [id#78L AS cliente_id#80L]
      +- Range (1, 501, step=1, splits=Some(2))

== Optimized Logical Plan ==
Join Inner, (cast(cliente#18 as bigint) = cliente_id#80L)
:- Filter isnotnull(cliente#18)
:  +- Relation [id_venda#17,cliente#18,categor

dfBroadcast_CSV: DataFrame = [id_venda: int, cliente: string ... 6 more fields]

In [11]:
//Sort Merge Join
// Desativamos o Broadcast e garantimos que o Shuffle Hash Join esteja desativado (padrão)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.join.preferSortMergeJoin", "true")

val dfSortMerge = dfVendas.join(dfClientes, "cliente_id")
dfSortMerge.explain(extended=true)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [cliente_id])
:- Project [venda_id#69L, cliente_id#71L, (cast(venda_id#69L as double) * 2.5) AS valor#74]
:  +- Project [venda_id#69L, ((venda_id#69L % cast(500 as bigint)) + cast(1 as bigint)) AS cliente_id#71L]
:     +- Project [id#67L AS venda_id#69L]
:        +- Range (1, 1000000, step=1, splits=Some(2))
+- Project [cliente_id#80L, concat(Cliente_, cast(cliente_id#80L as string)) AS nome#82]
   +- Project [id#78L AS cliente_id#80L]
      +- Range (1, 501, step=1, splits=Some(2))

== Analyzed Logical Plan ==
cliente_id: bigint, venda_id: bigint, valor: double, nome: string
Project [cliente_id#71L, venda_id#69L, valor#74, nome#82]
+- Join Inner, (cliente_id#71L = cliente_id#80L)
   :- Project [venda_id#69L, cliente_id#71L, (cast(venda_id#69L as double) * 2.5) AS valor#74]
   :  +- Project [venda_id#69L, ((venda_id#69L % cast(500 as bigint)) + cast(1 as bigint)) AS cliente_id#71L]
   :     +- Project [id#67L AS venda_id#69L]
   :       

dfSortMerge: DataFrame = [cliente_id: bigint, venda_id: bigint ... 2 more fields]

In [12]:
val df_udf = Seq(
  ("João", 25),
  ("Maria", 17),
  ("Carlos", 65)
).toDF("nome", "idade")

// Definindo a UDF
val classificarFaixaEtaria = udf((idade: Int) => {
  if (idade < 18) "Menor de idade"
  else if (idade < 60) "Adulto"
  else "Idoso"
})

// 2. Aplicando a UDF
val df_udf_resultado = df_udf.withColumn("faixa_etaria", classificarFaixaEtaria(col("idade")))

df_udf: DataFrame = [nome: string, idade: int]
classificarFaixaEtaria: org.apache.spark.sql.expressions.UserDefinedFunction = SparkUserDefinedFunction(
  f = ammonite.$sess.cmd12$Helper$$Lambda$6999/0x0000007001efb040@584d4311,
  dataType = StringType,
  inputEncoders = List(
    Some(
      value = ExpressionEncoder(
        objSerializer = BoundReference(
          ordinal = 0,
          dataType = IntegerType,
          nullable = false
        ),
        objDeserializer = AssertNotNull(
          child = UpCast(
            child = GetColumnByOrdinal(ordinal = 0, dataType = IntegerType),
            target = IntegerType,
            walkedTypePath = List("- root class: \"int\"")
          ),
          walkedTypePath = List("- root class: \"int\"")
        ),
        clsTag = Int
      )
    )
  ),
  outputEncoder = Some(
    value = ExpressionEncoder(
      objSerializer = StaticInvoke(
        staticObject = class org.apache.spark.unsafe.types.UTF8String,
        dataType = String

In [13]:
df_udf_resultado.explain(true)

== Parsed Logical Plan ==
'Project [nome#148, idade#149, UDF('idade) AS faixa_etaria#154]
+- Project [_1#143 AS nome#148, _2#144 AS idade#149]
   +- LocalRelation [_1#143, _2#144]

== Analyzed Logical Plan ==
nome: string, idade: int, faixa_etaria: string
Project [nome#148, idade#149, UDF(idade#149) AS faixa_etaria#154]
+- Project [_1#143 AS nome#148, _2#144 AS idade#149]
   +- LocalRelation [_1#143, _2#144]

== Optimized Logical Plan ==
LocalRelation [nome#148, idade#149, faixa_etaria#154]

== Physical Plan ==
LocalTableScan [nome#148, idade#149, faixa_etaria#154]

